# Profiling Messy CSVs with Dataprof

In data engineering, you often receive CSV files that contain mixed data types, missing values, and corrupted text. This exercise shows how to use `dataprof` to swiftly detect and summarize these issues.

In [ ]:
# Run this cell first to install dependencies in Colab
!pip install pandas dataprof==0.6.2

## Generate Messy Dummy Data

Let's create a terribly formatted CSV to simulate wild, unclean data. Notice the arbitrary date formats, missing objects, and mixed numeric/text fields.

In [1]:
import pandas as pd
import numpy as np

# Build the corrupted dataset elements
messy_data = {
    'user_id': ['101', '102', '103', 'UNKNOWN', '105', '106', '107', None, '109', '110'],
    'age': [25, 30, 'thirty-five', 40, np.nan, 22, -5, 29, 31, 899],
    'signup_date': ['2023-01-01', '2023-01-02', 'not_a_date', '2023/15/01', '2023-01-05', '01-06-2023', pd.NaT, '2023-01-08', '20230109', '2023-01-10'],
    'score': [8.5, 9.0, np.nan, 7.5, 8.0, 9.5, 6.0, 7.0, np.nan, 8.2]
}

df_messy = pd.DataFrame(messy_data)
df_messy.to_csv("messy_data.csv", index=False)
print("Created 'messy_data.csv'!")

display(df_messy)

Created 'messy_data.csv'!


,user_id,age,signup_date,score
0,101,25,2023-01-01,8.5
1,102,30,2023-01-02,9.0
2,103,thirty-five,not_a_date,NaN
3,UNKNOWN,40,2023/15/01,7.5
4,105,NaN,2023-01-05,8.0
5,106,22,01-06-2023,9.5
6,107,-5,NaT,6.0
7,NaN,29,2023-01-08,7.0
8,109,31,20230109,NaN
9,110,899,2023-01-10,8.2


## Profile the Data

We'll load the literal string versions of this CSV (which is how pandas usually falls back on messy data without schema pre-validation) and then use `dataprof` to expose the anomalies.

In [ ]:
import dataprof as dp

# 1. Load the data exactly as it appears in the wild
df_raw = pd.read_csv("messy_data.csv", dtype=str)

# 2. Hand it off to dataprof to crunch the underlying anomalies
report = dp.profile(df_raw)

# 3. Extract key metrics
print("--- DATA PROFILE SUMMARY ---")
print(f"Total Rows Diagnosed: {report.rows}")

print("\n--- CRITICAL COLUMN INSIGHTS ---")
# column_profiles is now a dict {name: ColumnProfile} — iterate its values
for col_profile in report.column_profiles.values():
    print(f"\nColumn: '{col_profile.name}'")
    print(f"  - Missing Value Rate: {col_profile.null_count} out of {report.rows}")

    # Numeric range
    if col_profile.min is not None:
        print(f"  - Range detected: {col_profile.min} to {col_profile.max}")

    # Text-length stats (0.6.2: min_length / max_length / avg_length on string columns)
    if col_profile.avg_length is not None:
        print(
            f"  - String lengths: min={col_profile.min_length}, "
            f"max={col_profile.max_length}, avg={col_profile.avg_length:.1f}"
        )

    # Detected value patterns (0.6.2: e.g. date, email, IP, …)
    if col_profile.patterns:
        best = max(col_profile.patterns, key=lambda p: p.match_count)
        print(f"  - Top pattern: {best.name} ({best.match_percentage:.0f}% of values)")

# 4. Quick statistical summary — like DataFrame.describe() but for the profile
print("\n--- STATISTICAL SUMMARY (describe) ---")
print(report.describe())

# 5. Quality score at a glance (0.6.2: quality_summary())
print("\n--- QUALITY SUMMARY ---")
print(report.quality_summary())

# 6. Write full schema out for automated CI/CD checks next time
# save() now supports .json, .csv, and .parquet (0.6.2)
print("\nSaving report to JSON and CSV for further inspection...")
report.save("messy_report.json").save("messy_report.csv")

--- DATA PROFILE SUMMARY ---
Total Rows Diagnosed: 10

--- CRITICAL COLUMN INSIGHTS ---

Column: 'user_id'
  - Missing Value Rate: 1 out of 10

Column: 'score'
  - Missing Value Rate: 2 out of 10

Column: 'age'
  - Missing Value Rate: 1 out of 10

Column: 'signup_date'
  - Missing Value Rate: 1 out of 10

Saving report to JSON for further inspection...


Column,Type,Count,Null %,Stats
user_id,string,10,10.0%,
score,string,10,20.0%,
age,string,10,10.0%,
signup_date,date,10,10.0%,
